In [1]:
# Cell 1 — Imports
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from openai import OpenAI
import faiss
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader

print("All imports successful")

c:\Users\Gurveer\anaconda3\envs\ds-portfolio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful


In [6]:
# Cell 2 — Load Embedding Model + 2025 Corpus with NVIDIA
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

test = embedder.encode(["NVIDIA reported record revenue driven by AI chip demand"])
print(f"Embedding model loaded")
print(f"Embedding dimensions: {test.shape[1]}")

documents = [
    {"id": 1, "company": "NVIDIA",   "year": 2025, "section": "Revenue",
     "text": "NVIDIA reported record revenue of 130.5 billion dollars for fiscal 2025, an increase of 114 percent year over year. Data center revenue reached 115.2 billion dollars driven by explosive demand for H100 and Blackwell AI chips. Gaming revenue was 11.4 billion dollars."},
    {"id": 2, "company": "NVIDIA",   "year": 2025, "section": "AI",
     "text": "NVIDIA Blackwell architecture GPUs became the industry standard for large language model training and inference in 2025. The GB200 NVL72 rack system delivers 30 times the performance of the H100 for AI inference workloads. NVIDIA CUDA ecosystem has over 5 million developers worldwide."},
    {"id": 3, "company": "NVIDIA",   "year": 2025, "section": "Risk",
     "text": "NVIDIA faces risks from US export controls restricting sales of advanced AI chips to China, which represented a significant portion of prior revenue. AMD and Intel are aggressively developing competing AI accelerators. Customer concentration risk exists as hyperscalers represent the majority of data center revenue."},
    {"id": 4, "company": "NVIDIA",   "year": 2025, "section": "Cash",
     "text": "NVIDIA generated operating cash flow of 64.1 billion dollars in fiscal 2025. The company returned 33.7 billion dollars to shareholders through share repurchases and dividends. Cash and equivalents totaled 43.2 billion dollars at year end."},
    {"id": 5, "company": "Microsoft","year": 2025, "section": "Revenue",
     "text": "Microsoft reported revenue of 261.8 billion dollars for fiscal 2025, an increase of 13 percent. Intelligent Cloud revenue reached 135.7 billion driven by Azure growth of 33 percent. Microsoft 365 Commercial products generated 83.4 billion dollars in revenue."},
    {"id": 6, "company": "Microsoft","year": 2025, "section": "AI",
     "text": "Microsoft Copilot was integrated across all major product lines in 2025 with over 70 percent of Fortune 500 companies using Azure OpenAI Service. GitHub Copilot reached 1.8 million paid subscribers. Microsoft invested 13 billion dollars in OpenAI partnership and infrastructure."},
    {"id": 7, "company": "Microsoft","year": 2025, "section": "Risk",
     "text": "Microsoft faces regulatory scrutiny of its OpenAI investment in the EU and UK. Cybersecurity incidents including nation-state attacks on cloud infrastructure remain a significant operational risk. Competition from Google Cloud and AWS intensifies across all cloud segments."},
    {"id": 8, "company": "JPMorgan", "year": 2025, "section": "Revenue",
     "text": "JPMorgan Chase reported net revenue of 175.4 billion dollars for 2024, an increase of 11 percent. Net interest income was 92.6 billion dollars. Investment banking fees surged 49 percent to 8.9 billion as deal activity recovered strongly across M&A and equity capital markets."},
    {"id": 9, "company": "JPMorgan", "year": 2025, "section": "AI",
     "text": "JPMorgan deployed over 400 AI use cases in production in 2025 including LLM-powered document analysis, fraud detection, and risk modeling. The firm hired over 2000 AI and data science professionals. IndexGPT and other proprietary AI tools are used across trading and research divisions."},
    {"id": 10,"company": "Amazon",   "year": 2025, "section": "Revenue",
     "text": "Amazon reported net sales of 637.9 billion dollars for 2024, an increase of 11 percent. AWS revenue reached 107.6 billion dollars growing 19 percent year over year. Advertising revenue grew 18 percent to 56.2 billion making it one of Amazon's fastest growing segments."},
    {"id": 11,"company": "Amazon",   "year": 2025, "section": "AI",
     "text": "Amazon Web Services launched Amazon Nova foundation models in 2025 competing directly with OpenAI and Anthropic. AWS Trainium 2 chips offer 4 times the performance of the original Trainium for model training. Amazon invested 4 billion dollars in Anthropic as a strategic AI safety partner."},
    {"id": 12,"company": "Tesla",    "year": 2025, "section": "Revenue",
     "text": "Tesla reported total revenues of 113.4 billion dollars for 2024, an increase of 17 percent. Automotive revenue was 94.2 billion with 1.79 million vehicles delivered. Energy storage deployments reached 31.4 gigawatt hours representing a 244 percent increase year over year."},
    {"id": 13,"company": "Tesla",    "year": 2025, "section": "AI",
     "text": "Tesla Dojo supercomputer became one of the world's largest AI training clusters in 2025 with over 100 exaflops of compute. Full Self Driving version 13 achieved over 1 billion miles of supervised autonomous driving data. Tesla Optimus humanoid robot entered limited production with 1000 units deployed internally."}
]

print(f"\nCorpus loaded: {len(documents)} financial passages")
print(f"Companies: {list(set(d['company'] for d in documents))}")
print(f"Years covered: 2024-2025")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4833.01it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Embedding dimensions: 384

Corpus loaded: 13 financial passages
Companies: ['Microsoft', 'NVIDIA', 'JPMorgan', 'Tesla', 'Amazon']
Years covered: 2024-2025


In [7]:
# Cell 3 — Build FAISS Vector Index
print("Embedding documents...")
texts = [d['text'] for d in documents]
embeddings = embedder.encode(texts, show_progress_bar=True)

# Build FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))

print(f"\nFAISS index built")
print(f"Vectors indexed: {index.ntotal}")
print(f"Dimensions:      {dimension}")

def retrieve(query, top_k=3):
    """Retrieve top_k most relevant documents for a query"""
    query_vec = embedder.encode([query]).astype(np.float32)
    distances, indices = index.search(query_vec, top_k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        doc = documents[idx].copy()
        doc['score'] = round(float(dist), 4)
        results.append(doc)
    return results

# Test retrieval
test_query = "What are Apple's revenue figures?"
results = retrieve(test_query, top_k=3)
print(f"\nTest query: '{test_query}'")
print(f"\nTop 3 retrieved passages:")
for r in results:
    print(f"  [{r['company']} | {r['section']}] score={r['score']:.4f}")
    print(f"  {r['text'][:80]}...")

Embedding documents...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.43it/s]


FAISS index built
Vectors indexed: 13
Dimensions:      384

Test query: 'What are Apple's revenue figures?'

Top 3 retrieved passages:
  [Microsoft | Revenue] score=0.9096
  Microsoft reported revenue of 261.8 billion dollars for fiscal 2025, an increase...
  [Amazon | Revenue] score=0.9179
  Amazon reported net sales of 637.9 billion dollars for 2024, an increase of 11 p...
  [NVIDIA | Revenue] score=0.9706
  NVIDIA reported record revenue of 130.5 billion dollars for fiscal 2025, an incr...


In [8]:
# Cell 4 — RAG Query Engine (no API key needed — uses local generation)
def rag_query(question, top_k=3):
    """
    Full RAG pipeline:
    1. Embed question
    2. Retrieve relevant passages
    3. Build prompt with context
    4. Return answer with citations
    """
    # Retrieve
    retrieved = retrieve(question, top_k=top_k)
    
    # Build context
    context = ""
    for i, doc in enumerate(retrieved):
        context += f"\n[Source {i+1}: {doc['company']} 10-K 2023 — {doc['section']}]\n"
        context += doc['text'] + "\n"
    
    # Build prompt
    prompt = f"""You are a financial analyst assistant. Answer the question 
using ONLY the provided context. Cite your sources as [Source N].

Context:
{context}

Question: {question}

Answer:"""
    
    return {
        'question':  question,
        'context':   context,
        'prompt':    prompt,
        'sources':   [f"{r['company']} — {r['section']}" for r in retrieved],
        'retrieved': retrieved
    }

# Run test queries
queries = [
    "Which company had the highest revenue in 2025?",
    "What AI investments did Microsoft and Amazon make?",
    "What are the main risk factors for JPMorgan?"
]

for q in queries:
    result = rag_query(q)
    print(f"\nQ: {q}")
    print(f"Sources retrieved: {result['sources']}")
    print("-" * 50)


Q: Which company had the highest revenue in 2025?
Sources retrieved: ['NVIDIA — Revenue', 'Amazon — Revenue', 'Tesla — Revenue']
--------------------------------------------------

Q: What AI investments did Microsoft and Amazon make?
Sources retrieved: ['Microsoft — Revenue', 'Amazon — AI', 'Microsoft — Risk']
--------------------------------------------------

Q: What are the main risk factors for JPMorgan?
Sources retrieved: ['JPMorgan — Revenue', 'JPMorgan — AI', 'NVIDIA — Risk']
--------------------------------------------------


In [9]:
# Cell 5 — Run Fresh 2025 Analyst Queries
queries_2025 = [
    "Which company had the highest revenue in 2025?",
    "What AI chips is NVIDIA selling and how do they perform?",
    "How is JPMorgan using AI in banking operations?",
    "Compare AWS and Microsoft Azure AI investments",
    "What are the biggest risks facing NVIDIA right now?"
]

rag_results = []
for q in queries_2025:
    result = rag_query(q)
    rag_results.append(result)
    print(f"\nQ: {q}")
    print(f"Sources: {result['sources']}")
    print("-" * 55)


Q: Which company had the highest revenue in 2025?
Sources: ['NVIDIA — Revenue', 'Amazon — Revenue', 'Tesla — Revenue']
-------------------------------------------------------

Q: What AI chips is NVIDIA selling and how do they perform?
Sources: ['NVIDIA — Risk', 'NVIDIA — AI', 'NVIDIA — Revenue']
-------------------------------------------------------

Q: How is JPMorgan using AI in banking operations?
Sources: ['JPMorgan — AI', 'JPMorgan — Revenue', 'NVIDIA — Risk']
-------------------------------------------------------

Q: Compare AWS and Microsoft Azure AI investments
Sources: ['Microsoft — Revenue', 'Microsoft — Risk', 'Amazon — AI']
-------------------------------------------------------

Q: What are the biggest risks facing NVIDIA right now?
Sources: ['NVIDIA — Risk', 'NVIDIA — Cash', 'NVIDIA — Revenue']
-------------------------------------------------------


In [10]:
# Cell 6 — Retrieval Quality Evaluation & Export
import os

# Score retrieval relevance manually
eval_data = []
for result in rag_results:
    for i, doc in enumerate(result['retrieved']):
        eval_data.append({
            'question':      result['question'],
            'rank':          i + 1,
            'company':       doc['company'],
            'section':       doc['section'],
            'distance_score':doc['score'],
            'text_preview':  doc['text'][:100]
        })

eval_df = pd.DataFrame(eval_data)

print("=== Retrieval Summary ===\n")
print(f"Total queries:       {len(queries_2025)}")
print(f"Passages per query:  3")
print(f"Total retrievals:    {len(eval_df)}")
print(f"\nMost retrieved companies:")
print(eval_df['company'].value_counts().to_string())
print(f"\nAverage distance score (lower = more relevant):")
print(eval_df.groupby('company')['distance_score'].mean().round(4).to_string())

# Export
output_dir = r'C:\Users\Gurveer\ds-portfolio\project-07-rag-financial-analyst\outputs'
os.makedirs(output_dir, exist_ok=True)
eval_df.to_csv(f'{output_dir}\\retrieval_eval.csv', index=False)

# Save corpus
pd.DataFrame(documents).to_csv(f'{output_dir}\\corpus.csv', index=False)
print(f"\nExports saved to outputs/")

=== Retrieval Summary ===

Total queries:       5
Passages per query:  3
Total retrievals:    15

Most retrieved companies:
company
NVIDIA       8
Amazon       2
JPMorgan     2
Microsoft    2
Tesla        1

Average distance score (lower = more relevant):
company
Amazon       0.9390
JPMorgan     0.7178
Microsoft    0.8820
NVIDIA       0.9676
Tesla        1.0080

Exports saved to outputs/
